# Q-Agent GRPO Training — Reinforcement Learning for Question Generation
Train the Q-Agent using **Group Relative Policy Optimization (GRPO)** with custom reward functions.

**Base model**: `unsloth/Qwen2.5-14B-Instruct`  
**Oracle model**: `unsloth/Llama-3.3-70B-Instruct` (loaded separately for judging + adversarial solving)  

**Reward signals (2 functions, 1 oracle call)**:
1. **Format + Length** (max 3.0) — valid JSON, required keys, 4 choices, length constraints (Q+options+answer ≤ 130 tokens, total ≤ 1024 tokens). **No LLM call — pure validation.**  
2. **Oracle Combined + Adversarial** (−3.0 to 8.0) — **single oracle call** where the oracle solves the question independently, verifies the marked answer is correct, plus evaluates topic alignment (±1.5), distractor quality (±1.5), difficulty (±2.0). If marked answer is wrong → big penalty (−3.0); if oracle solves correctly → penalty (−1.0); if oracle fails → reward (+3.0).

**Key idea**: The Q-Agent generates questions that are **hard for the oracle to solve** while ensuring **the marked answer is actually correct**. Only **1 oracle call per sample** for efficiency.

In [ ]:
import os
os.environ['HF_HOME'] = '/workspace/AAIPL/hf_models/'

from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

# ═══ Configuration ═══
max_seq_length = 2048   # Q-Agent generations can be long
lora_rank = 64          # Higher rank for GRPO
dtype = torch.bfloat16

# ── Load Q-Agent base model: Qwen2.5-14B-Instruct ──
QAGENT_MODEL = "unsloth/Qwen2.5-14B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=QAGENT_MODEL,
    max_seq_length=max_seq_length,
    load_in_4bit=False,       # Full precision on MI300X (192GB VRAM)
    fast_inference=True,       # Enable vLLM fast inference for GRPO
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,  # Reserve VRAM for oracle model
    dtype=dtype,
    trust_remote_code=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Q-Agent loaded: {QAGENT_MODEL}")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Load Oracle Model
The oracle (`Llama-3.3-70B-Instruct`, 4-bit) serves two purposes:

1. **Quality judge** — evaluates topic alignment, distractor quality, and difficulty  
2. **Adversarial solver** — tries to solve the question independently (without seeing the answer). If it succeeds, the Q-Agent is penalized; if it fails, the Q-Agent gets a bonus reward.

In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

# ══════════════════════════════════════════════════════════════
# Oracle model — for judging topic/difficulty/distractors + adversarial solving
# ══════════════════════════════════════════════════════════════

ORACLE_MODEL_NAME = "unsloth/Llama-3.3-70B-Instruct"
# ORACLE_MODEL_NAME = "unsloth/Qwen2.5-14B-Instruct"  # Smaller fallback

print(f"Loading oracle model: {ORACLE_MODEL_NAME}")
oracle_model, oracle_tokenizer = FastLanguageModel.from_pretrained(
    model_name=ORACLE_MODEL_NAME,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,   # 4-bit to save VRAM
    trust_remote_code=True,
)
FastLanguageModel.for_inference(oracle_model)
print(f"Oracle loaded: {ORACLE_MODEL_NAME}")
print(f"Oracle params: {sum(p.numel() for p in oracle_model.parameters()):,}")


def oracle_judge(prompt: str, max_new_tokens: int = 256) -> str:
    """Query the oracle model for a judgment."""
    messages = [
        {"role": "system", "content": "You are a precise evaluator. Answer exactly as instructed."},
        {"role": "user", "content": prompt},
    ]
    inputs = oracle_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(oracle_model.device)
    outputs = oracle_model.generate(
        **inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=True,
        top_p=0.9, pad_token_id=oracle_tokenizer.pad_token_id or oracle_tokenizer.eos_token_id,
    )
    return oracle_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print(f"\n✓ Oracle loaded for reward computation (judging + adversarial solving).")

## Prepare GRPO Dataset
GRPO needs a dataset of **prompts** (not prompt+response pairs). Each prompt asks the Q-Agent to generate a question on a specific topic. The model generates completions, and reward functions score them.

In [ ]:
import json
import random
from datasets import Dataset

# ── Topics and subtypes from the competition ──
TOPICS = {
    "Logical Reasoning": ["Syllogisms"],
    "Puzzles": ["Seating Arrangements (Linear, Circular)"],
    "Blood Relations and Family Tree": ["Family tree logic"],
    "Series and Patterns": ["Mixed Series (Alphanumeric)"],
}

# ── Per-subtype generation instructions (same as SFT training) ──
SUBTYPE_INSTRUCTIONS = {
    "both_neither_conclusion": "Generate a syllogism MCQ presenting two conclusions from given premises. The choices should be: Only I follows / Only II follows / Both follow / Neither follows.",
    "which_conclusion_follows": "Generate a syllogism MCQ with 4 conclusion options where the solver must identify which one logically follows from the premises.",
    "which_does_not_follow": "Generate a syllogism MCQ with 4 conclusion options where the solver must identify which one does NOT logically follow from the premises.",
    "numeric_next_term": "Generate a numeric series MCQ asking for the next term. Use patterns such as arithmetic, geometric, squares, cubes, Fibonacci, or mixed operations.",
    "alphanumeric_next_term": "Generate an alphanumeric series MCQ asking for the next term. Use patterns combining letters and numbers with systematic progression.",
    "missing_term": "Generate a series MCQ with a missing term in the middle (shown as '?'). The solver must identify the pattern and fill in the blank.",
    "odd_one_out": "Generate a series MCQ where one number is wrong. The solver must find the incorrect term that breaks the pattern.",
    "simple_relation_2hop": "Generate a simple blood relations MCQ involving 2 relationship hops (e.g., A's father's wife).",
    "moderate_relation_3hop": "Generate a moderate blood relations MCQ involving 3 relationship hops.",
    "complex_relation_4hop": "Generate a complex blood relations MCQ involving 4 relationship hops.",
    "extended_relation_5hop": "Generate a challenging blood relations MCQ involving 5 relationship hops.",
    "linear_position_query": "Generate a linear seating arrangement MCQ asking who sits at a specific position.",
    "linear_adjacent_query": "Generate a linear seating arrangement MCQ asking who sits immediately next to a person.",
    "circular_position_query": "Generate a circular seating arrangement MCQ asking who sits to the left or right of a person.",
    "circular_adjacent_query": "Generate a circular seating arrangement MCQ asking who sits next to a person at a circular table.",
    "circular_between_query": "Generate a circular seating arrangement MCQ asking who sits between two persons.",
}

QAGENT_SYSTEM_PROMPT = """You are a competitive question generator for logical reasoning challenges. 
Generate a challenging multiple-choice question (MCQ) in valid JSON format for the given topic.
The question must have exactly 4 choices (A, B, C, D) with exactly one correct answer.
Include a clear explanation of why the correct answer is right.

Output constraint:
You must output ONLY valid JSON.

JSON schema:
{
  "properties": {
    "topic": {"type": "string"},
    "question": {"type": "string"},
    "choices": {"type": "array", "items": {"type": "string"}, "minItems": 4, "maxItems": 4},
    "answer": {"enum": ["A", "B", "C", "D"], "type": "string"},
    "explanation": {"type": "string"}
  },
  "required": ["topic", "question", "choices", "answer", "explanation"],
  "type": "object"
}"""


def build_grpo_dataset(n_samples: int = 5000, seed: int = 42) -> Dataset:
    """Build a dataset of prompts for GRPO training."""
    random.seed(seed)
    
    all_subtopics = []
    for parent, children in TOPICS.items():
        for child in children:
            all_subtopics.append((parent, child))
    
    subtypes_by_topic = {
        "Syllogisms": ["both_neither_conclusion", "which_conclusion_follows", "which_does_not_follow"],
        "Mixed Series (Alphanumeric)": ["numeric_next_term", "alphanumeric_next_term", "missing_term", "odd_one_out"],
        "Family tree logic": ["simple_relation_2hop", "moderate_relation_3hop", "complex_relation_4hop", "extended_relation_5hop"],
        "Seating Arrangements (Linear, Circular)": ["linear_position_query", "linear_adjacent_query", "circular_position_query", "circular_adjacent_query", "circular_between_query"],
    }
    
    records = []
    for _ in range(n_samples):
        parent, child = random.choice(all_subtopics)
        topic_full = f"{parent}/{child}"
        
        # Pick a random subtype for this topic
        available_subtypes = subtypes_by_topic.get(child, [])
        if available_subtypes:
            subtype = random.choice(available_subtypes)
            instruction = SUBTYPE_INSTRUCTIONS.get(subtype, f"Generate a challenging MCQ on: {topic_full}")
        else:
            subtype = ""
            instruction = f"Generate a challenging MCQ on: {topic_full}"
        
        user_parts = [instruction, f"Topic: {topic_full}"]
        if subtype:
            user_parts.append(f"Question Type: {subtype}")
        
        prompt = [
            {"role": "system", "content": QAGENT_SYSTEM_PROMPT},
            {"role": "user", "content": "\n".join(user_parts)},
        ]
        
        records.append({
            "prompt": prompt,
            "topic": topic_full,
            "subtype": subtype,
        })
    
    return Dataset.from_list(records)


dataset = build_grpo_dataset(n_samples=5000)
print(f"GRPO dataset: {len(dataset)} prompts")
print(f"Sample prompt:")
for msg in dataset[0]["prompt"]:
    print(f"  [{msg['role']}]: {msg['content'][:150]}...")
print(f"\nTopic distribution:")
from collections import Counter
topic_dist = Counter(dataset["topic"])
for t, c in sorted(topic_dist.items()):
    print(f"  {t}: {c}")

## Reward Functions (2 functions, 1 oracle call per sample)

| # | Function | Max | LLM call? | What it checks |
|---|----------|-----|-----------|----------------|
| R1 | `format_length_reward_func` | 3.0 | No | JSON validity, required keys, 4 choices, token length ≤ 130 / ≤ 1024 |
| R2 | `oracle_combined_reward_func` | 8.0 | 1 oracle call | Oracle **solves** the question, **verifies** marked answer, plus topic match, distractor quality, difficulty |

**Answer verification** (inside R2): Oracle checks if the marked answer is actually correct.
- Marked answer is **wrong** → **−3.0** (flat penalty, skip all other scoring)

**Adversarial logic** (inside R2, only if answer is valid): Oracle solves independently.
- Oracle answer matches marked answer → **−1.0** (too easy, oracle solved it)
**Reward range**: −3.0 to 11.0

- Oracle answer doesn't match → **+3.0** (hard question, oracle failed)

In [ ]:
import re

def parse_question_json(text: str) -> dict | None:
    """Extract JSON object from Q-Agent output."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass
    
    match = re.search(r"\{[\s\S]*\}", text)
    if match:
        try:
            parsed = json.loads(match.group())
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass
    return None


def count_tokens(text: str) -> int:
    """Count tokens using the Q-Agent tokenizer."""
    return len(tokenizer.encode(text, add_special_tokens=False))


# ═══════════════════════════════════════════════════════════════
# REWARD 1: Format + Length (max 3.0) — NO oracle call
# ═══════════════════════════════════════════════════════════════

def format_length_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    Structural validity + token length constraints.
    
    Format (max 2.0):
      +0.5  valid JSON
      +0.3  has all required keys (topic, question, choices, answer, explanation)
      +0.4  exactly 4 choices with A)/B)/C)/D) prefix
      +0.3  answer is a single letter in {A, B, C, D}
      +0.3  explanation is non-empty (>10 chars)
      +0.2  question is non-trivial (>20 chars)
    
    Length (max 1.0):
      +0.5  question + options + answer ≤ 130 tokens
      +0.5  question + options + answer + explanation ≤ 1024 tokens
      (penalty: −0.5 each if exceeded)
    """
    rewards = []
    for completion in completions:
        content = completion[0]["content"] if isinstance(completion, list) else completion
        reward = 0.0
        
        parsed = parse_question_json(content)
        if parsed is None:
            rewards.append(0.0)
            continue
        
        # ── Format checks (max 2.0) ──
        reward += 0.5  # Valid JSON
        
        required = ["topic", "question", "choices", "answer", "explanation"]
        if all(k in parsed for k in required):
            reward += 0.3
        else:
            rewards.append(reward)
            continue
        
        choices = parsed.get("choices", [])
        if (isinstance(choices, list) and len(choices) == 4 and
            all(isinstance(c, str) and len(c) > 2 and c[0] in "ABCD" and c[1] == ")" 
                for c in choices)):
            reward += 0.4
        
        answer = parsed.get("answer", "")
        if isinstance(answer, str) and len(answer) == 1 and answer in "ABCD":
            reward += 0.3
        
        explanation = parsed.get("explanation", "")
        if isinstance(explanation, str) and len(explanation) > 10:
            reward += 0.3
        
        question = parsed.get("question", "")
        if isinstance(question, str) and len(question) > 20:
            reward += 0.2
        
        # ── Length checks (max 1.0) ──
        choices_text = "\n".join(choices) if isinstance(choices, list) else ""
        q_plus_opts = f"{question}\n{choices_text}\n{answer}"
        q_plus_all  = f"{q_plus_opts}\n{explanation}"
        
        tokens_short = count_tokens(q_plus_opts)
        tokens_full  = count_tokens(q_plus_all)
        
        if tokens_short <= 130:
            reward += 0.5   # Q + options + answer fits in 130 tokens
        else:
            reward -= 0.5   # Penalty: too long for competition format
        
        if tokens_full <= 1024:
            reward += 0.5   # Total fits in 1024 tokens
        else:
            reward -= 0.5   # Penalty: total too long
        
        rewards.append(max(reward, 0.0))  # Floor at 0
    
    return rewards


# ═══════════════════════════════════════════════════════════════
# REWARD 2: Oracle Combined + Adversarial (−1.0 to 8.0) — SINGLE oracle call
#   Oracle SOLVES the question (does NOT see the answer), plus
#   evaluates topic, distractor quality, difficulty.
# ═══════════════════════════════════════════════════════════════

def oracle_combined_reward_func(prompts, completions, topic, **kwargs) -> list[float]:
    """
    One oracle call per sample. The oracle receives the question and choices
    but NOT the marked answer, so it must solve independently.
    
    Oracle returns JSON:
      - your_answer: "A"/"B"/"C"/"D"      → adversarial: match = −1.0, mismatch = +3.0
      - topic_match: "YES"/"PARTIAL"/"NO"  → 1.5 / 0.75 / 0.0
      - distractor_quality: 1-5            → 0.3 – 1.5
      - difficulty: 1-5                    → 0.4 – 2.0
    
    Max total: 8.0  |  Min: −1.0
    """
    rewards = []
    for i, completion in enumerate(completions):
        content = completion[0]["content"] if isinstance(completion, list) else completion
        parsed = parse_question_json(content)
        
        if parsed is None:
            rewards.append(0.0)
            continue
        
        question_text = parsed.get("question", "")
        choices = parsed.get("choices", [])
        marked_answer = parsed.get("answer", "")
        generated_topic = parsed.get("topic", "")
        requested_topic = topic[i] if isinstance(topic, list) else topic
        
        # Skip oracle if question is structurally broken
        if (not question_text or not choices or len(choices) != 4 or
            marked_answer not in "ABCD" or len(marked_answer) != 1):
            rewards.append(0.0)
            continue
        
        # ── Build ONE combined oracle prompt (NO marked answer shown) ──
        choices_str = "\n".join(choices)
        combined_prompt = (
            f"You are evaluating a generated multiple-choice question.\n\n"
            f"Requested topic: \"{requested_topic}\"\n\n"
            f"Question: {question_text}\n"
            f"Options:\n{choices_str}\n\n"
            f"Tasks:\n"
            f"1. Solve the question yourself — pick the best answer (A, B, C, or D).\n"
            f"2. Does the question match the requested topic?\n"
            f"3. Rate the distractor (wrong option) quality from 1-5.\n"
            f"4. Rate the question difficulty from 1-5.\n\n"
            f"Respond ONLY with this exact JSON format:\n"
            f'{{"your_answer": "A or B or C or D", "topic_match": "YES or PARTIAL or NO", '
            f'"distractor_quality": 1, "difficulty": 1}}'
        )
        
        try:
            response = oracle_judge(combined_prompt, max_new_tokens=100)
            
            # Parse oracle JSON response
            oracle_result = None
            try:
                oracle_result = json.loads(response.strip())
            except json.JSONDecodeError:
                m = re.search(r"\{[^}]+\}", response)
                if m:
                    try:
                        oracle_result = json.loads(m.group())
                    except json.JSONDecodeError:
                        pass
            
            reward = 0.0
            
            if oracle_result and isinstance(oracle_result, dict):
                # ── Adversarial: oracle tries to solve (−1.0 to +3.0) ──
                oracle_answer = str(oracle_result.get("your_answer", "")).strip().upper()
                # Normalize: take first letter if multi-char
                if oracle_answer and oracle_answer[0] in "ABCD":
                    oracle_answer = oracle_answer[0]
                else:
                    oracle_answer = None
                
                if oracle_answer is None:
                    reward += 2.0   # Oracle couldn't parse/answer → likely confusing question
                elif oracle_answer == marked_answer:
                    reward += -1.0  # Oracle solved it → penalty (too easy)
                else:
                    reward += 3.0   # Oracle got it wrong → reward (hard question!)
                
                # ── Topic match (max 1.5) ──
                topic_match = str(oracle_result.get("topic_match", "")).upper()
                if "YES" in topic_match:
                    reward += 1.5
                elif "PARTIAL" in topic_match:
                    reward += 0.75
                
                # ── Distractor quality (max 1.5) ──
                dq = oracle_result.get("distractor_quality", 0)
                if isinstance(dq, (int, float)) and 1 <= dq <= 5:
                    reward += dq * 0.3  # 0.3 – 1.5
                
                # ── Difficulty (max 2.0) ──
                diff = oracle_result.get("difficulty", 0)
                if isinstance(diff, (int, float)) and 1 <= diff <= 5:
                    reward += diff * 0.4  # 0.4 – 2.0
            else:
                # Oracle response unparseable — use heuristic for topic only
                topic_lower = requested_topic.lower()
                q_lower = question_text.lower()
                kw_map = {
                    "syllogism": ["statement", "conclusion", "premise", "syllogism"],
                    "blood": ["father", "mother", "brother", "sister", "son", "daughter", "uncle", "family"],
                    "family": ["father", "mother", "brother", "sister", "son", "daughter", "uncle", "family"],
                    "seating": ["seated", "sitting", "left", "right", "row", "circle", "table"],
                    "arrangement": ["seated", "sitting", "left", "right", "row", "circle", "table"],
                    "series": ["series", "next term", "pattern", "sequence", "find the"],
                    "pattern": ["series", "next term", "pattern", "sequence", "find the"],
                }
                for key, words in kw_map.items():
                    if key in topic_lower and any(w in q_lower for w in words):
                        reward += 0.75  # Partial topic credit
                        break
            
            rewards.append(reward)
        
        except Exception:
            rewards.append(0.0)
    
    return rewards


print("✓ All 2 reward functions defined (1 oracle call per sample):")
print("  1. format_length_reward_func        (max 3.0) — Format + token length checks (no LLM)")
print("  2. oracle_combined_reward_func      (−1.0 to 8.0) — Adversarial solving + topic + distractor + difficulty (1 oracle call)")
print(f"\nMax total reward: 11.0  |  Min: −1.0  |  Oracle calls per sample: 1")
print(f"\nAdversarial logic: oracle solves WITHOUT seeing the answer")
print(f"  Oracle correct → −1.0 (too easy)  |  Oracle wrong → +3.0 (hard question!)")

## GRPO Training Configuration

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    use_vllm=True,                       # vLLM fast inference for generation
    learning_rate=5e-6,                   # Conservative LR for RL
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,        # GRPO: 1 prompt per step
    gradient_accumulation_steps=4,        # Accumulate for stability
    num_generations=8,                    # Generate 8 completions per prompt for GRPO
    max_prompt_length=512,                # System + user prompt
    max_completion_length=1024,           # Q-Agent output can be long (JSON + explanation)
    # num_train_epochs=1,                 # Uncomment for full epoch run
    max_steps=500,                        # Start with 500; increase if reward is climbing
    save_steps=100,
    max_grad_norm=0.1,
    report_to="none",                     # Set to "wandb" for W&B logging
    output_dir="qagent_grpo_outputs",
    bf16=True,
)

print(f"GRPO Config:")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Max completion length: {training_args.max_completion_length}")
print(f"  Max steps: {training_args.max_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_length_reward_func,        # R1: Format + length     (max 3.0, no LLM)
        oracle_combined_reward_func,      # R2: Oracle combined     (−1.0 to 8.0, 1 oracle call)
    ],
    args=training_args,
    train_dataset=dataset,
)

print("✓ GRPOTrainer initialized")
print(f"  Reward functions: 2 (1 oracle call per sample)")
print(f"  Dataset size: {len(dataset)}")
print(f"  Expected reward range: −1.0 — 11.0")

## Train
Run the GRPO training loop. Watch the `reward` column — it should increase over time.

**Expected behavior:**
- Steps 1-50: Format reward dominates; oracle adversarial reward may be negative (easy questions)
- Steps 50-200: Model learns valid JSON + correct topics; adversarial reward starts climbing  
- Steps 200-500: Q-Agent generates harder questions that the oracle fails to solve; difficulty + adversarial rewards increase  

If reward plateaus early, consider increasing `max_steps` or adjusting reward weights.

In [ ]:
trainer.train()

## Save Model

In [ ]:
# Save LoRA adapters
model.save_lora("qagent_grpo_lora")
print("LoRA saved to: qagent_grpo_lora/")

# Save merged 16-bit model
model.save_pretrained_merged(
    "qagent_grpo_merged_16bit", tokenizer, save_method="merged_16bit"
)
print("Merged model saved to: qagent_grpo_merged_16bit/")

## Test Inference
Generate sample questions with the GRPO-trained Q-Agent and compare quality.

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=1024,
)

test_prompts = [
    {
        "topic": "Logical Reasoning/Syllogisms",
        "instruction": "Generate a syllogism MCQ presenting two conclusions from given premises.",
    },
    {
        "topic": "Blood Relations and Family Tree/Family tree logic",
        "instruction": "Generate a complex blood relations MCQ involving 4 relationship hops.",
    },
    {
        "topic": "Puzzles/Seating Arrangements (Linear, Circular)",
        "instruction": "Generate a circular seating arrangement MCQ asking who sits between two persons.",
    },
    {
        "topic": "Series and Patterns/Mixed Series (Alphanumeric)",
        "instruction": "Generate an alphanumeric series MCQ asking for the next term.",
    },
]

print("=" * 60)
print("GRPO-Trained Q-Agent — Sample Generations")
print("=" * 60)

for test in test_prompts:
    messages = [
        {"role": "system", "content": QAGENT_SYSTEM_PROMPT},
        {"role": "user", "content": f"{test['instruction']}\nTopic: {test['topic']}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Generate WITHOUT LoRA (base SFT model)
    output_base = model.fast_generate(
        [text], sampling_params=sampling_params, lora_request=None,
    )[0].outputs[0].text
    
    # Generate WITH GRPO LoRA
    output_grpo = model.fast_generate(
        [text], sampling_params=sampling_params,
        lora_request=model.load_lora("qagent_grpo_lora"),
    )[0].outputs[0].text
    
    print(f"\n{'─' * 60}")
    print(f"Topic: {test['topic']}")
    print(f"\n[BASE SFT]:  {output_base[:400]}")
    print(f"\n[GRPO]:      {output_grpo[:400]}")

print(f"\n{'═' * 60}")

## Evaluate Quality: Both Rewards

Run both reward functions on a batch of generations. One oracle call per sample (quality + adversarial).

In [ ]:
import numpy as np
from collections import defaultdict

# Generate a batch of questions for evaluation
eval_prompts = []
for _ in range(50):
    parent, child = random.choice(list(
        (p, c) for p, cs in TOPICS.items() for c in cs
    ))
    topic_full = f"{parent}/{child}"
    eval_prompts.append({
        "messages": [
            {"role": "system", "content": QAGENT_SYSTEM_PROMPT},
            {"role": "user", "content": f"Generate a challenging MCQ.\nTopic: {topic_full}"},
        ],
        "topic": topic_full,
    })

sampling_params_eval = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024)

# Generate with GRPO LoRA
texts = [
    tokenizer.apply_chat_template(p["messages"], tokenize=False, add_generation_prompt=True)
    for p in eval_prompts
]
outputs = model.fast_generate(
    texts, sampling_params=sampling_params_eval,
    lora_request=model.load_lora("qagent_grpo_lora"),
)

# Score each generation with both reward functions
scores = defaultdict(list)
for i, (output, prompt_data) in enumerate(zip(outputs, eval_prompts)):
    content = output.outputs[0].text
    completion_wrapped = [[{"content": content}]]
    prompt_wrapped = [prompt_data["messages"]]
    topic_wrapped = [prompt_data["topic"]]
    
    r1 = format_length_reward_func(prompt_wrapped, completion_wrapped[0])
    r2 = oracle_combined_reward_func(prompt_wrapped, completion_wrapped[0], topic=topic_wrapped)
    
    scores["fmt+len"].append(r1[0])
    scores["oracle+adv"].append(r2[0])
    scores["total"].append(r1[0] + r2[0])

print(f"\n{'═' * 60}")
print(f"GRPO Q-Agent Evaluation (n={len(eval_prompts)})")
print(f"{'═' * 60}")
print(f"{'Reward':>15s}  {'Mean':>7s}  {'Std':>7s}  {'Min':>7s}  {'Max':>7s}")
print(f"{'─' * 50}")
for name in ["fmt+len", "oracle+adv", "total"]:
    vals = np.array(scores[name])
    print(f"{name:>15s}  {vals.mean():>7.3f}  {vals.std():>7.3f}  {vals.min():>7.3f}  {vals.max():>7.3f}")

# Summary metrics
good_format = sum(1 for s in scores["fmt+len"] if s >= 2.5)
print(f"\nGood format+length (≥2.5/3.0): {good_format}/{len(eval_prompts)} ({100*good_format/len(eval_prompts):.0f}%)")
high_oracle = sum(1 for s in scores["oracle+adv"] if s >= 5.0)
print(f"High oracle score (≥5.0/8.0):  {high_oracle}/{len(eval_prompts)} ({100*high_oracle/len(eval_prompts):.0f}%)")
oracle_fooled = sum(1 for s in scores["oracle+adv"] if s >= 3.0)
print(f"Oracle fooled (adversarial ≥3.0): {oracle_fooled}/{len(eval_prompts)} ({100*oracle_fooled/len(eval_prompts):.0f}%)")